# Construye tu primer agente de IA con LangChain

**Taller práctico · EurusConf 2026**

Vas a construir tres cosas: tu primer agente, un agente que investiga en la web con memoria, y tu propio agente de valor.

> **VS Code:** sigue el README (entorno + `.env`). **Colab:** corre la celda de instalación y pega tu API key cuando te la pida.

## 0. Setup

In [ ]:
# Solo en Google Colab (en VS Code ya instalaste con requirements.txt):
%pip install -q "langchain>=1.0" langchain-google-genai langchain-openai langchain-community ddgs python-dotenv pydantic requests

In [ ]:
# Carga tu API key y prepara el modelo (funciona en VS Code y en Colab)
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # lee .env si existe (VS Code)

PROVIDER = os.getenv("MODEL_PROVIDER", "google_genai")
MODEL_NAME = os.getenv("MODEL_NAME", "gemini-2.0-flash")

if PROVIDER == "google_genai" and not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Pega tu GOOGLE_API_KEY: ")

def get_model():
    if PROVIDER == "openrouter":  # OpenRouter usa la API compatible con OpenAI
        return init_chat_model(f"openai:{MODEL_NAME}",
                               base_url="https://openrouter.ai/api/v1",
                               api_key=os.getenv("OPENROUTER_API_KEY"))
    return init_chat_model(f"{PROVIDER}:{MODEL_NAME}")

model = get_model()
print("Modelo listo:", PROVIDER, MODEL_NAME)

In [ ]:
# Smoke test: si esto responde, estás listo
print(model.invoke("Responde en una sola frase: ¿qué es un agente de IA?").content)

## Teoría mínima

- Un **agente** = modelo + herramientas + un *loop*: recibe una tarea, decide si usar una herramienta, observa el resultado y repite hasta responder.
- **Patrones de diseño:** tool use (lo de hoy), reflexión, planning, multiagente, RAG.
- **Ecosistema:** `create_agent` (hoy) corre sobre **LangGraph**; **Deep Agents** es un harness más completo; **LangSmith** observa y evalúa.

## Práctica 1 — Tu primer agente

Un agente con una herramienta. El agente decide cuándo usarla; el loop la ejecuta solo.

In [ ]:
# DEMO: tu primer agente
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def hora_actual(zona: str = "local") -> str:
    """Devuelve la fecha y hora actual."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M")

agente = create_agent(
    model=model,
    tools=[hora_actual],
    system_prompt="Eres un asistente útil. Usa las herramientas cuando ayuden.",
)
r = agente.invoke({"messages": [{"role": "user", "content": "¿Qué fecha y hora es?"}]})
print(r["messages"][-1].content)

### Fundamento: salida estructurada
En vez de texto suelto, pide datos validados (útil para integrar).

In [ ]:
from pydantic import BaseModel, Field

class Tarea(BaseModel):
    titulo: str = Field(description="qué hay que hacer")
    prioridad: str = Field(description="alta, media o baja")

estructurado = model.with_structured_output(Tarea)
print(estructurado.invoke("Tengo que entregar el informe de física mañana, es urgente"))

### Tu turno
Crea una herramienta `calcular` y un agente que use dos herramientas.

In [ ]:
# TODO: completa la herramienta para que evalúe una expresión, ej. "12 * 8"
@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática simple."""
    pass  # TODO

# TODO: crea un agente con hora_actual Y calcular, y pregúntale algo que use 'calcular'

### Solución P1

In [ ]:
@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática simple, ej. '12 * 8'."""
    try:
        return str(eval(expresion, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

agente2 = create_agent(model=model, tools=[hora_actual, calcular],
                       system_prompt="Eres un asistente útil.")
r = agente2.invoke({"messages": [{"role": "user", "content": "¿Cuánto es el 15% de 240?"}]})
print(r["messages"][-1].content)

## Práctica 2 — Tu agente investigador web (con memoria)

Le damos una herramienta que lo conecta a internet, y memoria para que recuerde la conversación.

In [ ]:
# DEMO: agente que busca en la web
from langchain_community.tools import DuckDuckGoSearchRun

buscar_web = DuckDuckGoSearchRun()
investigador = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres un investigador. Busca en la web y responde con información actual.",
)
r = investigador.invoke({"messages": [{"role": "user", "content": "¿Qué es LangChain y para qué se usa?"}]})
print(r["messages"][-1].content)

### Dale memoria
Con un `checkpointer` y un `thread_id`, el agente recuerda la conversación.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agente_memoria = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres un asistente que recuerda la conversación.",
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "demo-1"}}
agente_memoria.invoke({"messages": [{"role": "user", "content": "Me interesan los agentes de IA."}]}, config)
r = agente_memoria.invoke({"messages": [{"role": "user", "content": "¿Qué tema te dije que me interesa?"}]}, config)
print(r["messages"][-1].content)  # debería recordarlo

### Tu turno
Enfoca el agente a TU tema (carrera, hobby) cambiando el `system_prompt`.

In [ ]:
# TODO: crea tu investigador con un system_prompt enfocado a tu tema y hazle una pregunta actual

### Solución P2

In [ ]:
mi_investigador = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres un investigador experto en tecnología. Responde claro y con datos actuales.",
)
r = mi_investigador.invoke({"messages": [{"role": "user", "content": "¿Qué novedades hay en inteligencia artificial?"}]})
print(r["messages"][-1].content)

## ¿Por qué construir tu propio agente si Claude Code ya existe?

Para tareas generales, usa lo que ya existe. Construyes el TUYO cuando tienes un flujo específico y repetible que se beneficia de **control**, **integración a tus sistemas**, **costo/escala** y **automatización real**. Ahora lo haces tuyo.

## Práctica 3 — Tu agente de valor

Construyes el tuyo. **Requisitos:** usa `create_agent` y los fundamentos, dale **memoria**, usa **salida estructurada**, y elige **un patrón** y el problema que resuelves.

Elige UNA opción (abajo tienes el ejemplo de cada una) y escribe tu versión en la celda **"TU CÓDIGO AQUÍ"**. Necesitas el token de tu servicio en `.env` (ver `.env.example`).

- **A — Notion:** organiza tus notas/tareas.
- **B — Todoist:** gestiona tu lista de tareas inmediata.
- **C — API pública:** conéctalo a un servicio que te interese (clima, noticias, finanzas...).

### Opción A — Notion

In [ ]:
import os, requests
from langgraph.checkpoint.memory import InMemorySaver

@tool
def agregar_a_notion(titulo: str) -> str:
    """Agrega una nota o tarea a tu base de datos de Notion."""
    resp = requests.post(
        "https://api.notion.com/v1/pages",
        headers={"Authorization": f"Bearer {os.getenv('NOTION_TOKEN')}",
                 "Notion-Version": "2022-06-28",
                 "Content-Type": "application/json"},
        json={"parent": {"database_id": os.getenv("NOTION_DATABASE_ID")},
              "properties": {"Name": {"title": [{"text": {"content": titulo}}]}}},
    )
    return "Agregado a Notion" if resp.ok else f"Error: {resp.text[:120]}"

agente_notion = create_agent(model=model, tools=[agregar_a_notion],
    system_prompt="Ayudas a organizar notas y tareas en Notion.",
    checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "notion"}}
r = agente_notion.invoke({"messages": [{"role": "user", "content": "Agrega 'estudiar para el parcial' a mi Notion"}]}, cfg)
print(r["messages"][-1].content)

### Opción B — Todoist

In [ ]:
import os, requests
from langgraph.checkpoint.memory import InMemorySaver

@tool
def agregar_a_todoist(tarea: str) -> str:
    """Agrega una tarea a tu Todoist."""
    resp = requests.post(
        "https://api.todoist.com/rest/v2/tasks",
        headers={"Authorization": f"Bearer {os.getenv('TODOIST_TOKEN')}"},
        json={"content": tarea},
    )
    return "Tarea agregada a Todoist" if resp.ok else f"Error: {resp.text[:120]}"

agente_todoist = create_agent(model=model, tools=[agregar_a_todoist],
    system_prompt="Ayudas a gestionar la lista de tareas en Todoist.",
    checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "todoist"}}
r = agente_todoist.invoke({"messages": [{"role": "user", "content": "Agrega 'entregar laboratorio el viernes'"}]}, cfg)
print(r["messages"][-1].content)

### Opción C — API pública (ejemplo: clima)

In [ ]:
import os, requests
from langgraph.checkpoint.memory import InMemorySaver

@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad."""
    resp = requests.get("https://api.openweathermap.org/data/2.5/weather",
        params={"q": ciudad, "appid": os.getenv("OPENWEATHER_API_KEY"), "units": "metric", "lang": "es"})
    if not resp.ok:
        return f"Error: {resp.text[:120]}"
    d = resp.json()
    return f"{ciudad}: {d['weather'][0]['description']}, {d['main']['temp']}C"

agente_clima = create_agent(model=model, tools=[clima],
    system_prompt="Informas el clima y sugieres qué hacer.",
    checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "clima"}}
r = agente_clima.invoke({"messages": [{"role": "user", "content": "¿Cómo está el clima en Ciudad de Panamá?"}]}, cfg)
print(r["messages"][-1].content)

### Tu turno — tu agente de valor

In [ ]:
# =========================================================
#  TU CÓDIGO AQUÍ
#  Requisitos:
#   1. Usa create_agent con al menos una herramienta (tool).
#   2. Dale memoria: checkpointer=InMemorySaver() + thread_id.
#   3. Usa salida estructurada (Pydantic) en algún punto.
#   4. Elige un patrón y el problema que resuelves.
#  Ya tienes a mano: create_agent, tool, InMemorySaver, BaseModel, Field, requests, model
# =========================================================

## Cierre

Construiste tu primer agente, un investigador con memoria, y tu propio agente de valor.

**¿Cuándo usar qué?** `create_agent` para la mayoría; **LangGraph** para control total del flujo; **Deep Agents** para tareas largas y complejas. **LangSmith** para observar y evaluar.

Recursos: https://docs.langchain.com · MART Automations: https://martautomations.com